In [16]:
import pandas as pd
import sqlite3

df = pd.read_csv("BoCo_200.csv")

df = df.dropna(subset=['volunteer_id']) # remove rows where volunteer_id = NaN
df['volunteer_id'] = df['volunteer_id'].astype(int).astype(str).str.zfill(3) # convert volunteer_id into numerical entries

mask = df['Referral'] == 'N'
df.loc[mask, 'ref_name'] = 'N/A'

# adding ability to apply code to more than 2 volunteering events
n_events = df.filter(regex=r'^event_date\.\d+$').shape[1]

# reformating of csv file entries
for i in range(1, n_events + 1):
    event = f"event_date.{i}"
    sign_up = f"sign_up.{i}"
    attend = f"attended.{i}"

# formating date entries and naming days of the week for future use
    dt = pd.to_datetime(df[event])
    df[event] = dt.dt.strftime("%Y-%m-%d")

    df[f"day_of_week.{i}"] = dt.dt.day_name()

# cleaning up attendance entries
    mask = (df[sign_up] == 'N') & (df[attend].isna())
    df.loc[mask, attend] = 'N/A'

# formating time
for i in range(1, n_events + 1):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"

    dt_in = pd.to_datetime(df[time_in], format="%I:%M %p", errors='coerce')
    dt_out = pd.to_datetime(df[time_out], format="%I:%M %p", errors='coerce')
    df[time_in] = dt_in.dt.strftime("%I:%M %p")
    df[time_out] = dt_out.dt.strftime("%I:%M %p")

# cleaning up empty time entries
for i in range(1, n_events + 1):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"

    mask = (df[time_in].isna())
    df.loc[mask, [time_in]] = 'N/A'

    mask = (df[time_out].isna())
    df.loc[mask, [time_out]] = 'N/A'

# no case match and spacing differences
df['ref_name'] = df['ref_name'].str.strip().str.title()

# calculating individual volunteer hours per event
for i in range(1, n_events + 1):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"
    hours = f"hours.{i}"

    df[hours] = (
        pd.to_datetime(df[time_out], format="%I:%M %p", errors='coerce')
        - pd.to_datetime(df[time_in], format="%I:%M %p", errors='coerce')
    ).dt.total_seconds() / 3600 # converts seconds to hours

    df[hours] = df[hours].fillna(0)

# swap to long format to be able to use in tableau
stubs = ['event_date', 'sign_up', 'attended', 'time_in',
         'time_out', 'day_of_week', 'hours']

long_df = pd.wide_to_long(df, stubnames=stubs, i='volunteer_id',
                          j='event', sep='.').reset_index()

long_df[['time_in', 'time_out']] = long_df[['time_in', 'time_out']].replace('N/A', None)
long_df = long_df.sort_values(['volunteer_id', 'event'])

# storing data
conn = sqlite3.connect('BoCo.db')

df.to_sql('volunteer_log', conn, if_exists='replace', index=False)
long_df.to_sql('volunteer_events', conn, if_exists='replace', index=False)

check = pd.read_sql('SELECT * FROM volunteer_log LIMIT 5', conn)
print(check)

conn.close()

  volunteer_id name_first    name_last                         email  \
0          001      Jane          Doe             jane.doe@gmail.com   
1          002      John          Doe             john.doe@gmail.com   
2          003    Emmett     Hutchins      emmett.hutchins@gmail.com   
3          004      Ezra        Novak           ezra.novak@gmail.com   
4          005   Camille   Achterberg   camille.achterberg@gmail.com   

               phone Referral  ref_name event_date.1 sign_up.1 attended.1  \
0  +1 (123) 456-7890        Y  Isabelle   2027-01-01         Y          Y   
1  +1 (321) 456-7891        N       N/A   2027-01-01         N        N/A   
2  +1 (225) 959-4506        Y    Adrian   2027-01-01         Y          Y   
3  +1 (484) 206-2615        N       N/A   2027-01-01         Y          Y   
4  +1 (500) 843-5925        N       N/A   2027-01-01         Y          Y   

   ... time_out.1 event_date.2 sign_up.2 attended.2 time_in.2 time_out.2  \
0  ...   12:00 PM   2027-01-